# 1장 2강: 좋은 지표의 조건 — 실습문제

## 실습 목표

- Ravenstack의 비즈니스 목표에 맞춰 KPI, 보조 지표, 허무 지표를 구분할 수 있다.
- 구독 데이터에서 현재 반복 매출과 이탈률을 계산할 수 있다.
- 요금제 또는 유입 경로별 지표를 비교하여 개선이 필요한 대상을 찾을 수 있다.
- 행동 가능성, 비교 가능성, 이해 용이성을 기준으로 지표의 적절성을 판단할 수 있다.

## 실습 환경 / 데이터

- Python
- pandas
- `ravenstack_accounts.csv`
- `ravenstack_subscriptions.csv`

Ravenstack은 기업 고객에게 구독형 소프트웨어를 제공하는 B2B SaaS 서비스입니다.

이번 실습에서는 Ravenstack의 핵심 목표를 다음과 같이 가정합니다.

> **유료 구독을 안정적으로 유지하면서 반복 매출을 늘린다.**

주요 컬럼은 다음과 같습니다.

| 테이블 | 컬럼 | 의미 |
|---|---|---|
| accounts | `account_id` | 고객사 식별자 |
| accounts | `referral_source` | 고객사가 유입된 경로 |
| accounts | `signup_date` | 고객사 가입일 |
| accounts | `churn_flag` | 고객사 이탈 여부 |
| subscriptions | `subscription_id` | 구독 식별자 |
| subscriptions | `plan_tier` | 구독 요금제 |
| subscriptions | `mrr_amount` | 월간 반복 매출(MRR) |
| subscriptions | `is_trial` | 체험 구독 여부 |
| subscriptions | `end_date` | 구독 종료일 |
| subscriptions | `churn_flag` | 구독 이탈 여부 |

> 비율은 별도 지시가 없으면 소수점 둘째 자리의 백분율로 출력합니다.

## 실습 준비

1. 필요한 라이브러리를 불러오세요.
2. 두 CSV 파일을 각각 `accounts`, `subscriptions`에 불러오세요.
3. 각 데이터의 행과 열 개수, 상위 5개 행을 확인하세요.
4. 분석 대상 컬럼의 자료형과 결측치 개수를 확인하세요.
5. `signup_date`, `start_date`, `end_date`를 날짜형으로 변환하세요.
6. `mrr_amount`의 기초 통계량과 `plan_tier`의 빈도를 확인하세요.

In [1]:
import pandas as pd

accounts = pd.read_csv('./ravenstack_accounts.csv')
subscriptions = pd.read_csv('./ravenstack_subscriptions.csv')

print(accounts.shape)
print(subscriptions.shape)
display(accounts.head())
display(subscriptions.head())

print(accounts.dtypes)
print(accounts.isna().sum())
print(subscriptions.dtypes)
print(subscriptions.isna().sum())

accounts['signup_date'] = pd.to_datetime(accounts['signup_date'])
subscriptions['start_date'] = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date'] = pd.to_datetime(subscriptions['end_date'])

print(subscriptions['mrr_amount'].describe())
print(subscriptions['plan_tier'].value_counts())


(500, 10)
(5000, 14)


,account_id,account_name,industry,country,signup_date,referral_source,plan_tier,seats,is_trial,churn_flag
0,A-2e4581,Company_0,EdTech,US,2024-10-16,partner,Basic,9,False,False
1,A-43a9e3,Company_1,FinTech,IN,2023-08-17,other,Basic,18,False,True
2,A-0a282f,Company_2,DevTools,US,2024-08-27,organic,Basic,1,False,False
3,A-1f0ac7,Company_3,HealthTech,UK,2023-08-27,other,Basic,24,True,False
4,A-ce550d,Company_4,HealthTech,US,2024-10-27,event,Enterprise,35,False,True


,subscription_id,account_id,start_date,end_date,plan_tier,seats,mrr_amount,arr_amount,is_trial,upgrade_flag,downgrade_flag,churn_flag,billing_frequency,auto_renew_flag
0,S-8cec59,A-3c1a3f,2023-12-23,2024-04-12,Enterprise,14,2786,33432,False,False,False,True,monthly,True
1,S-0f6f44,A-9b9fe9,2024-06-11,NaN,Pro,17,833,9996,False,False,False,False,monthly,True
2,S-51c0d1,A-659280,2024-11-25,NaN,Enterprise,62,0,0,True,True,False,False,annual,False
3,S-f81687,A-e7a1e2,2024-11-23,2024-12-13,Enterprise,5,995,11940,False,False,False,True,monthly,True
4,S-cff5a2,A-ba6516,2024-01-10,NaN,Enterprise,27,5373,64476,False,False,False,False,monthly,True


account_id           str
account_name         str
industry             str
country              str
signup_date          str
referral_source      str
plan_tier            str
seats              int64
is_trial            bool
churn_flag          bool
dtype: object
account_id         0
account_name       0
industry           0
country            0
signup_date        0
referral_source    0
plan_tier          0
seats              0
is_trial           0
churn_flag         0
dtype: int64
subscription_id        str
account_id             str
start_date             str
end_date               str
plan_tier              str
seats                int64
mrr_amount           int64
arr_amount           int64
is_trial              bool
upgrade_flag          bool
downgrade_flag        bool
churn_flag            bool
billing_frequency      str
auto_renew_flag       bool
dtype: object
subscription_id         0
account_id              0
start_date              0
end_date             4514
plan_tier        

---

## 필수 1. 비즈니스 목표에 맞는 지표 구분하기

### 문제 1-1. 현재 반복 매출을 중심으로 지표를 계산하고 분류하기

#### 문제 설명

Ravenstack은 **유료 구독을 안정적으로 유지하면서 반복 매출을 늘리는 것**을 핵심 목표로 정했습니다.

다음 세 지표를 직접 계산한 뒤 현재 목표를 기준으로 KPI, 보조 지표, 허무 지표로 분류하세요.

- 현재 유료 구독 MRR
- 전체 구독 이탈률
- 누적 가입 고객사 수

이번 실습에서는 `end_date`가 비어 있고 `is_trial`이 `False`인 구독을 **현재 활성 유료 구독**으로 정의합니다.

#### 요구사항

1. 현재 활성 유료 구독 조건을 불리언 변수 `active_paid_mask`로 만드세요.
2. 현재 활성 유료 구독의 `mrr_amount` 합계를 `current_paid_mrr`로 계산하세요.
3. `subscriptions`의 `churn_flag` 평균으로 `subscription_churn_rate`를 계산하세요.
4. `accounts`의 고유한 `account_id` 개수를 `cumulative_accounts`로 계산하세요.
5. 세 지표를 알아보기 쉬운 형식으로 출력하세요.
6. 현재 비즈니스 목표를 기준으로 다음과 같이 분류하고 이유를 설명하세요.
   - 현재 유료 구독 MRR: KPI
   - 전체 구독 이탈률: 보조 지표
   - 누적 가입 고객사 수: 허무 지표 후보

#### 해석 질문

**Q1.** 현재 유료 구독 MRR을 KPI로 볼 수 있는 이유는 무엇인가요?  
**Q2.** 전체 구독 이탈률은 현재 유료 구독 MRR을 이해하는 데 어떻게 도움을 주나요?  
**Q3.** 누적 가입 고객사 수만으로 현재 서비스가 성장한다고 판단하기 어려운 이유는 무엇인가요?

#### 제출 결과

- 세 지표의 계산 코드와 결과
- KPI, 보조 지표, 허무 지표 분류
- 분류 이유
- Q1~Q3 답변

In [2]:
# 현재 활성 유료 구독 = 아직 안 끝났고(end_date 없음) trial도 아닌 구독
active_paid_mask = subscriptions['end_date'].isna() & (subscriptions['is_trial'] == False)

current_paid_mrr = subscriptions.loc[active_paid_mask, 'mrr_amount'].sum()
subscription_churn_rate = subscriptions['churn_flag'].mean()
cumulative_accounts = accounts['account_id'].nunique()

print('현재 유료 구독 MRR:', f'{current_paid_mrr:,}')
print('전체 구독 이탈률:', f'{subscription_churn_rate*100:.2f}%')
print('누적 가입 고객사 수:', cumulative_accounts)


현재 유료 구독 MRR: 10,159,608
전체 구독 이탈률: 9.72%
누적 가입 고객사 수: 500


세 지표를 목표(유료 구독 유지 + 반복 매출 증가) 기준으로 분류해보면 다음과 같다.

- 현재 유료 구독 MRR → KPI: trial 빼고 지금 실제로 돈 내고 있는 구독만 더한 값이라, 목표에서 말하는 "반복 매출" 자체를 거의 그대로 보여준다고 생각한다.
- 전체 구독 이탈률 → 보조 지표: 이탈률 자체가 목표는 아니지만, MRR이 왜 늘었는지 줄었는지 설명해주는 배경 지표로 쓸 수 있다.
- 누적 가입 고객사 수 → 허무 지표 후보: 이탈해도 절대 줄지 않고 계속 쌓이기만 하는 숫자라서, 지금 서비스가 잘 유지되고 있는지랑은 상관없이 커진다. 그래서 이걸 성과 지표처럼 쓰면 위험할 것 같다.


### 필수 1 답변 작성란

- **Q1.** 목표가 "유료 구독을 유지하면서 반복 매출을 늘리는 것"이기 때문에, trial과 이미 끝난 구독을 빼고 지금 실제로 결제되고 있는 MRR만 더한 `current_paid_mrr`이 이 목표를 가장 직접적으로 보여준다고 생각한다. 그래서 KPI로 분류했다.
- **Q2.** 이탈률은 MRR이 왜 그렇게 움직였는지 설명해주는 역할을 한다. 예를 들어 MRR이 줄었을 때 신규 유료 전환이 적어서인지, 원래 있던 구독이 이탈해서인지를 이탈률로 구분해볼 수 있어서 KPI를 해석하는 데 도움이 되는 보조 지표라고 생각한다.
- **Q3.** 누적 가입 고객사 수는 한 번 가입하면 나중에 이탈해도 숫자가 줄지 않고 계속 쌓이기만 한다. 그래서 실제로는 이탈이 많이 나서 상황이 안 좋아지고 있어도 이 숫자만 보면 계속 늘어나는 것처럼 보일 수 있어서, 이것만으로 성장했다고 판단하기는 어렵다.


---

## 필수 2. 요금제별 핵심 지표 비교하기

### 문제 2-1. 어느 요금제를 우선적으로 살펴봐야 하는가?

#### 문제 설명

전체 평균만 확인하면 요금제별 차이를 놓칠 수 있습니다. `Basic`, `Pro`, `Enterprise` 요금제별로 구독 규모, 이탈률, 현재 유료 구독 MRR을 비교하세요.

#### 요구사항

1. `subscriptions`에 `is_active_paid` 컬럼을 만드세요.
   - `end_date`가 비어 있고 `is_trial`이 `False`이면 `True`
2. `plan_tier`별로 다음 값을 집계하여 `plan_metrics`를 만드세요.
   - 전체 구독 수
   - 이탈 구독 수
   - 구독 이탈률
   - 현재 활성 유료 구독 수
3. 현재 활성 유료 구독만 사용해 요금제별 MRR 합계를 계산하고 `active_mrr` 컬럼으로 추가하세요.
4. 이탈률은 백분율로, MRR은 천 단위 구분 기호를 사용하여 출력하세요.
5. 현재 유료 구독 MRR이 가장 큰 요금제와 이탈률이 가장 높은 요금제를 확인하세요.
6. 현재 목표를 고려하여 우선적으로 점검할 요금제 하나를 정하고, 데이터 근거와 확인할 개선 방향을 설명하세요.

#### 해석 질문

**Q1.** 요금제별 이탈률을 비교하는 것이 전체 이탈률만 확인하는 것보다 행동 가능성이 높은 이유는 무엇인가요?  
**Q2.** 현재 유료 구독 MRR이 가장 큰 요금제는 무엇인가요?  
**Q3.** 이탈률이 가장 높은 요금제는 무엇인가요?  
**Q4.** 위 두 결과를 함께 보면 어떤 요금제를 우선적으로 점검할 수 있으며, 그 이유는 무엇인가요?

#### 제출 결과

- `plan_metrics` 집계 코드와 결과
- MRR 및 이탈률 기준 요금제 비교
- 우선 점검 대상과 개선 방향
- Q1~Q4 답변

In [3]:
subscriptions['is_active_paid'] = subscriptions['end_date'].isna() & (subscriptions['is_trial'] == False)

plan_metrics = subscriptions.groupby('plan_tier').agg(
    total_subscriptions=('subscription_id', 'count'),
    churned_subscriptions=('churn_flag', 'sum'),
    active_paid_subscriptions=('is_active_paid', 'sum')
)
plan_metrics['churn_rate'] = plan_metrics['churned_subscriptions'] / plan_metrics['total_subscriptions']

active_mrr = subscriptions.loc[subscriptions['is_active_paid']].groupby('plan_tier')['mrr_amount'].sum()
plan_metrics['active_mrr'] = active_mrr

plan_metrics_print = plan_metrics.copy()
plan_metrics_print['churn_rate'] = (plan_metrics_print['churn_rate']*100).round(2).astype(str) + '%'
plan_metrics_print['active_mrr'] = plan_metrics_print['active_mrr'].apply(lambda x: f'{x:,.0f}')
print(plan_metrics_print)

print('MRR 제일 큰 요금제:', plan_metrics['active_mrr'].idxmax())
print('이탈률 제일 높은 요금제:', plan_metrics['churn_rate'].idxmax())


            total_subscriptions  churned_subscriptions  \
plan_tier                                                
Basic                      1602                    152   
Enterprise                 1723                    172   
Pro                        1675                    162   

            active_paid_subscriptions churn_rate active_mrr  
plan_tier                                                    
Basic                            1228      9.49%    687,914  
Enterprise                       1304      9.98%  7,546,876  
Pro                              1282      9.67%  1,924,818  
MRR 제일 큰 요금제: Enterprise
이탈률 제일 높은 요금제: Enterprise


Enterprise가 MRR도 제일 크고 이탈률도 제일 높다. 요금제 간 이탈률 차이 자체는 그렇게 크지 않은데, Enterprise는 구독당 금액이 커서 같은 비율로 이탈해도 손실 금액이 훨씬 크다. 그래서 매출 관점에서 Enterprise를 먼저 점검하는 게 맞다고 생각한다. 온보딩이나 담당자 응대, 계약 갱신 프로세스 같은 부분을 좀 더 살펴보면 좋을 것 같다.


### 필수 2 답변 작성란

- **Q1.** 전체 이탈률만 보면 "그냥 이탈률이 9.72%구나" 정도만 알 수 있는데, 요금제별로 나눠보면 어떤 요금제에서 문제가 있는지 구체적으로 알 수 있다. 그래야 그 요금제에 맞는 조치(가격 조정, 전담 매니저 배치 등)를 실제로 취할 수 있어서 행동 가능성이 더 높다고 생각한다.
- **Q2.** MRR이 가장 큰 요금제는 Enterprise다.
- **Q3.** 이탈률이 가장 높은 요금제도 Enterpris다.
- **Q4.** 두 결과를 같이 보면 Enterprise가 매출도 제일 크고 이탈률도 제일 높아서 매출 손실 위험이 여기 몰려있다고 볼 수 있다. 그래서 Enterprise를 우선 점검 대상으로 정했고, 이탈 원인을 좀 더 들여다봐야 할 것 같다.


---

## 과제 1. 평가 문항 기반 독립 과제

### 문제 3-1. 유입 경로별 고객사 이탈률 비교하기

#### 문제 설명

Ravenstack 마케팅팀은 고객사가 유입된 경로에 따라 이탈 수준이 다른지 확인하려고 합니다. 유입 경로별 고객사 수와 이탈률을 비교하여 우선 점검할 유입 경로를 찾으세요.

> 이 과제는 필수 2에서 수행한 그룹별 지표 집계와 해석을 새로운 컬럼에 동일하게 적용하는 문제입니다.

#### 요구사항

1. `accounts`를 `referral_source`별로 그룹화하세요.
2. 다음 값을 집계하여 `source_metrics`를 만드세요.
   - 전체 고객사 수
   - 이탈 고객사 수
   - 고객사 이탈률
3. 고객사 이탈률이 높은 순서로 정렬하세요.
4. 이탈률은 소수점 둘째 자리의 백분율로 출력하세요.
5. 이탈률이 가장 높은 유입 경로를 찾으세요.
6. 해당 유입 경로를 우선 점검 대상으로 정하고, 확인할 개선 방향을 한 가지 제안하세요.
7. 유입 경로별 이탈률을 좋은 지표의 세 조건으로 평가하세요.
   - 행동 가능성
   - 비교 가능성
   - 이해 용이성

#### 해석 질문

**Q1.** 고객사 이탈률이 가장 높은 유입 경로는 무엇인가요?  
**Q2.** 유입 경로별 이탈률은 마케팅팀의 행동으로 어떻게 연결할 수 있나요?  
**Q3.** 유입 경로별 이탈률은 좋은 지표의 세 조건을 충족하나요?
**Q4.** KPI·보조 지표·허무 지표는 어떻게 다른가요? 고객사 이탈을 줄이려는 목적에서 각각의 지표 예시와 선정 이유를 제시하세요. 허무 지표는 어떤 맥락에서 성과 판단에 도움이 되지 않는지도 설명하세요.

#### 제출 결과

- `source_metrics` 집계 코드와 결과
- 우선 점검할 유입 경로
- 개선 방향
- 좋은 지표의 조건 평가
- Q1~Q4 답변

In [4]:
source_metrics = accounts.groupby('referral_source').agg(
    total_accounts=('account_id', 'count'),
    churned_accounts=('churn_flag', 'sum')
)
source_metrics['account_churn_rate'] = source_metrics['churned_accounts'] / source_metrics['total_accounts']
source_metrics = source_metrics.sort_values('account_churn_rate', ascending=False)

source_metrics_print = source_metrics.copy()
source_metrics_print['account_churn_rate'] = (source_metrics_print['account_churn_rate']*100).round(2).astype(str) + '%'
print(source_metrics_print)

print('이탈률 제일 높은 유입 경로:', source_metrics.index[0])


                 total_accounts  churned_accounts account_churn_rate
referral_source                                                     
event                        96                29             30.21%
other                       103                25             24.27%
ads                          98                23             23.47%
organic                     114                20             17.54%
partner                      89                13             14.61%
이탈률 제일 높은 유입 경로: event


event로 유입된 고객사의 이탈률이 30.21%로 다른 경로보다 눈에 띄게 높다. 이벤트로 들어온 고객이 실제로 필요해서 가입한 게 아니라 그 자리 분위기에 이끌려 가입했을 가능성도 있을 것 같아서, event 유입 고객의 온보딩 완료율이나 초기 기능 사용률을 따로 확인해보면 좋을 것 같다.

좋은 지표의 세 조건으로 평가하면:
- 행동 가능성: 마케팅팀이 채널별 예산/캠페인을 조정할 수 있는 대상이라 행동으로 이어지기 쉽다.
- 비교 가능성: 모든 경로를 똑같은 방식(이탈 고객사 수 ÷ 전체 고객사 수)으로 계산했으니 경로끼리 비교 가능하다.
- 이해 용이성: 비율이라서 누구나 바로 이해할 수 있다. 다만 경로별 고객사 수가 90~110개 정도라 표본이 크지 않다는 점은 감안해야 할 것 같다.


### 과제 1 답변 작성란

- **Q1.** 고객사 이탈률이 가장 높은 유입 경로는 event다.
- **Q2.** 어떤 채널에서 들어온 고객이 잘 이탈하는지 알면, 마케팅팀이 그 채널의 캠페인 타겟팅을 다시 검토하거나, 그 채널로 들어온 고객에게 온보딩을 더 신경 써서 제공하는 식으로 구체적인 행동을 정할 수 있다.
- **Q3.** 어느 정도 충족한다고 생각한다. 채널 단위로 조치할 수 있고(행동 가능성), 같은 방식으로 계산해서 비교할 수 있고(비교 가능성), 비율이라 이해하기 쉽다(이해 용이성). 다만 경로별 표본 수가 크지 않아서 우연히 차이가 났을 수도 있다는 점은 추가로 확인해봐야 할 것 같다.
- **Q4.** 고객사 이탈을 줄이는 걸 목표로 잡으면,
  - KPI: 전체 고객사 이탈률 — 목표를 제일 직접적으로 보여주는 지표라서 KPI로 잡았다.
  - 보조 지표: 유입 경로별/요금제별 이탈률 — KPI가 왜 그렇게 나왔는지, 어디서 이탈이 몰리는지 설명해주는 지표라서 보조 지표로 봤다.
  - 허무 지표: 누적 가입 고객사 수 — 이탈이 늘어나도 이 숫자는 절대 줄지 않고 계속 쌓이기만 해서, 이탈을 줄이자는 목적에는 맞지 않는 지표라고 생각한다.


---

## 실습 마무리

아래 질문에 답하세요.

1. 이번 실습에서 해결하려고 한 비즈니스 문제는 무엇인가요?
2. 현재 목표를 직접 보여주는 KPI로 어떤 지표를 선택했나요?
3. KPI의 변화를 이해하기 위해 어떤 보조 지표를 확인했나요?
4. 허무 지표 후보를 핵심 성과로 사용할 때 어떤 문제가 생길 수 있나요?
5. 어떤 분석 결과를 근거로 개선 대상을 정했나요?

### 실습 마무리 답변

1. 이번 실습에서는 Ravenstack의 목표인 "유료 구독을 유지하면서 반복 매출을 늘리는 것"을 어떤 지표로 봐야 하는지, 그리고 그 지표를 요금제나 유입 경로별로 나눠보면 뭐가 보이는지를 확인해봤다.
2. KPI로는 현재 유료 구독 MRR을 선택했다. trial이랑 이미 끝난 구독을 빼고 지금 실제로 나오는 매출만 계산한 값이라서 목표를 가장 잘 나타낸다고 생각했다.
3. KPI를 이해하기 위한 보조 지표로 전체 구독 이탈률, 그리고 이걸 더 쪼갠 요금제별 이탈률/MR, 유입 경로별 이탈률을 확인했다.
4. 누적 가입 고객사 수를 허무 지표 후보로 다뤘다. 이탈이 나도 절대 줄지 않는 숫자라서 실제 상황을 잘 못 보여줄 수 있다는 게 문제인 것 같다.
5. 요금제 비교에서는 Enterprise가 매출도 제일 크고 이탈률도 제일 높아서 우선 점검 대상으로 잡았고, 유입 경로 비교에서는 **event**로 들어온 고객사의 이탈률이 제일 높아서 이것도 우선 점검 대상으로 잡았다.
